# Notebook Docente 03 — Resolución espacial y temporal (Bloque 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-01/docente/colab/03_resoluciones_espacial_temporal.ipynb)

## Teledetección — Maestría en Ingeniería, Universidad del Magdalena

> ⚠️ **Este notebook es una DEMO DEL DOCENTE** — durante la Sesión 1 el docente lo proyecta
> en pantalla mientras explica la teoría. Los estudiantes NO lo corren en clase.
>
> **Para estudiantes:** puedes abrirlo después de la sesión para repasar. Todo está comentado en español.
> No necesitas ejecutar nada — puedes leer las celdas y los resultados de la demostración.

---

**Sesión 1 · Viernes 17 de julio de 2026**

---

**Este notebook es para que TÚ (el docente) lo proyectes en pantalla.** Versión en
Python/Colab del script `gee/03_resoluciones_espacial_temporal.js`.

### Qué vas a mostrar
1. La misma zona cacaotera vista con Sentinel-2 (10 m), Landsat 8 (30 m) y MODIS (250 m)
   en mapas lado a lado — para que los estudiantes vean con sus propios ojos qué significa
   "perder detalle".
2. Cuántas imágenes de cada satélite existen realmente en el mismo mes — la resolución
   temporal deja de ser un número en una tabla y se vuelve un conteo real.

In [1]:
!pip install geemap --quiet
print("✓ Instalación completada")

✓ Instalación completada


In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='teledeteccion-miguepoloc')  # reemplaza con tu ID de proyecto GEE

print("✓ Google Earth Engine inicializado correctamente")

✓ Google Earth Engine inicializado correctamente


## Parte A — Resolución espacial: tres satélites, la misma zona

In [3]:
zona_cacaotera = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])
fecha_inicio, fecha_fin = '2024-01-01', '2024-03-31'

# Sentinel-2 (10 m)
sentinel2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_cacaotera)
    .filterDate(fecha_inicio, fecha_fin)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(zona_cacaotera)
)

# Landsat 8 (30 m) — reescalar con los factores de Collection 2, Level 2
def reescalar_landsat(img):
    opticas = img.select('SR_B.').multiply(0.0000275).add(-0.2)
    return img.addBands(opticas, None, True)

landsat = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(zona_cacaotera)
    .filterDate(fecha_inicio, fecha_fin)
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .map(reescalar_landsat)
    .median()
    .clip(zona_cacaotera)
)

# MODIS (250 m)
modis = (
    ee.ImageCollection('MODIS/061/MOD09GA')
    .filterBounds(zona_cacaotera)
    .filterDate(fecha_inicio, fecha_fin)
    .median()
    .clip(zona_cacaotera)
)

print("✓ Las tres colecciones están listas")

✓ Las tres colecciones están listas


In [4]:
# geemap.linked_maps crea una grilla de mapas sincronizados en zoom y desplazamiento
imagenes = [sentinel2, landsat, modis]
etiquetas = ['Sentinel-2 — 10 m', 'Landsat 8 — 30 m', 'MODIS — 250 m']
parametros_vis = [
    {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 4000},
    {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min': 0, 'max': 0.4},
    {'bands': ['sur_refl_b02', 'sur_refl_b01', 'sur_refl_b04'], 'min': -100, 'max': 4000},
]

mapa_comparativo = geemap.linked_maps(
    rows=1, cols=3,
    height='400px',
    center=[10.75, -74.0], zoom=12,
    ee_objects=imagenes,
    vis_params=parametros_vis,
    labels=etiquetas,
)
mapa_comparativo

GridspecLayout(children=(Output(layout=Layout(grid_area='widget001')), Output(layout=Layout(grid_area='widget0…

**En clase:** haz zoom hacia adentro en los tres mapas a la vez (están sincronizados) y
pregunta en qué imagen dejan de distinguirse los polígonos de las fincas individuales.

## Parte B — Resolución temporal: ¿cuántas imágenes hay realmente en un mes?

In [5]:
mes_inicio, mes_fin = '2024-02-01', '2024-03-01'

n_sentinel2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_cacaotera).filterDate(mes_inicio, mes_fin).size().getInfo()
)
n_landsat89 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
    .filterBounds(zona_cacaotera).filterDate(mes_inicio, mes_fin).size().getInfo()
)
n_modis = (
    ee.ImageCollection('MODIS/061/MOD09GA')
    .filterBounds(zona_cacaotera).filterDate(mes_inicio, mes_fin).size().getInfo()
)

print('=== Imágenes disponibles en 1 mes (feb-2024) sobre la zona cacaotera ===')
print('Sentinel-2 (A+B, revisita teórica 5 días):', n_sentinel2)
print('Landsat 8+9 combinados (revisita teórica 8 días):', n_landsat89)
print('MODIS (revisita teórica 1-2 días):', n_modis)

=== Imágenes disponibles en 1 mes (feb-2024) sobre la zona cacaotera ===
Sentinel-2 (A+B, revisita teórica 5 días): 24
Landsat 8+9 combinados (revisita teórica 8 días): 14
MODIS (revisita teórica 1-2 días): 29


**Pregunta para la clase:** con estos números reales, ¿por qué MODIS "ve" tantas veces más
seguido pero nunca se usa para monitorear una finca individual? (Bloque 5: el trade-off
resolución espacial vs. temporal — 250 m es demasiado grueso para distinguir una parcela).